# Spark X2.6 native multimodal formal full fine-tune

This is the formal source-bound campaign notebook. It implements SigLIP2 So400m NaFlex with tiles and spatial downshuffle, V-JEPA2.1 motion, Whisper/CLAP audio input, Qwen native speech output, Stable Audio native sound/music output, full-parameter stages, two bounded MiMo instructor-guided replay comparisons, resumable Hugging Face checkpoints and private HF model publication.

**Select G4 directly and keep the same VM.** Setup, preflight, qualification and training use one isolated CUDA environment. Keep model/data random I/O on VM-local disk. The released dataset has304,861 original instructions; complete bounded music-score views produce315,359 supervised views before source-group splitting. The local media cache is23GB; the free floor is44GB. Only two complete checkpoint slots can coexist while saving; after the new HF snapshot and its Drive receipt are verified, older local checkpoints are purged and the newest remains. A full Git LFS dataset clone is prohibited by this disk design.

| Purpose | Destination | Local behavior |
|---|---|---|
| Pinned training data | `datasets/TassP0lar/SparkX2.6-MM` | Bounded rolling shard cache |
| Complete resumable training state | Private `TassP0lar/SparkX2.6-MM-checkpoints` | Upload, verify, retain newest local snapshot |
| Completed inference model | Private `TassP0lar/SparkX2.6-MM` | Export and upload after final evaluation |
| Receipts, reports and instructor ledger | Your cold Drive project | Small closed uploads, no training I/O |

**Cell sequence:** G4 setup → metadata/access preflight → qualification → `RUN_MODE='train'` on this same VM → `RUN_MODE='resume'` after a later interruption. Recovery runs in a new subprocess; no deliberate VM replacement or CPU runtime switch is required. Keep emitted receipts; the same-VM profile is loaded automatically. Upload failures have an upload-only retry cell below. This is the complete staged script; no modality is removed to turn it into a trial.

**Evidence boundary:** the source/CPU component tests do not certify the complete CUDA model. This notebook must earn its whole-model and new-process recovery gates. Recovery records whether the VM actually changed; same-VM recovery is never labeled fresh-VM proof. A single notebook does not guarantee one uninterrupted VM session or successful multimodal quality within1000CU. It never silently reduces the curriculum to fit.

Default final destination is the **private model repository** `TassP0lar/SparkX2.6-MM`; the public dataset namespace is separate. No instructor call or training happens merely by opening this notebook. Read the downloaded `RUN.md` after bootstrap.

The runtime is downloaded from a pinned commit of [BaiTian6641/Spark-X2.6-VL](https://github.com/BaiTian6641/Spark-X2.6-VL). Its SHA256 and exact source inventory are checked before execution. The notebook contains no embedded source archive. Both CPU and CUDA setup recover an earlier Python-without-pip environment without relying on system `ensurepip`. Setup preserves existing files; training reclaims only older verified campaign checkpoints under the policy above.

In [ ]:
# Verified source bootstrap: standard library only, no credentials.
import hashlib, io, json, os, pathlib, sys, urllib.request, zipfile
SOURCE_SHA256 = '7df35726d7a16d8fb032b517aa1231efc9b168433712c57af93ede7bb7381148'
SOURCE_COMMIT = '4e90988168355a8a2de7025ed3dbf3bae5804cca'
SOURCE_URL = 'https://raw.githubusercontent.com/BaiTian6641/Spark-X2.6-VL/4e90988168355a8a2de7025ed3dbf3bae5804cca/releases/spark-formal-runtime.zip'
SOURCE_BYTES = 5021404
# A distinct source directory allows repair alongside the previous notebook.
SOURCE_ROOT = pathlib.Path('/content/spark-formal-source') / SOURCE_SHA256[:16]
request = urllib.request.Request(SOURCE_URL, headers={'User-Agent':'Spark-Colab-launcher/2'})
with urllib.request.urlopen(request, timeout=60) as response:
    _source_bytes = response.read(SOURCE_BYTES + 1)
if len(_source_bytes) != SOURCE_BYTES or hashlib.sha256(_source_bytes).hexdigest() != SOURCE_SHA256:
    raise RuntimeError('GitHub runtime size or SHA256 mismatch; no runtime source was executed')
with zipfile.ZipFile(io.BytesIO(_source_bytes)) as z:
    names = z.namelist()
    assert len(names) == len(set(names))
    manifest = json.loads(z.read('SOURCE_MANIFEST.json'))
    assert set(names) == {f['path'] for f in manifest['files']} | {'SOURCE_MANIFEST.json'}
    assert sum(f['bytes'] for f in manifest['files']) < 16*1024**2
    for f in manifest['files']:
        relative = pathlib.PurePosixPath(f['path'])
        assert not relative.is_absolute() and '..' not in relative.parts
        target = SOURCE_ROOT / relative
        assert target.resolve().is_relative_to(SOURCE_ROOT.resolve()) and not target.is_symlink()
        payload = z.read(f['path'])
        assert len(payload) == f['bytes'] and hashlib.sha256(payload).hexdigest() == f['sha256']
        if target.exists(): assert target.read_bytes() == payload, 'Changed source: use a new reviewed release'
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(payload)
    (SOURCE_ROOT/'SOURCE_MANIFEST.json').write_bytes(z.read('SOURCE_MANIFEST.json'))
del _source_bytes
print('Reviewed source:', SOURCE_SHA256)
print('GitHub commit:', SOURCE_COMMIT)
print((SOURCE_ROOT/'colab/final/RUN.md').read_text())


## Operator configuration

Keep Secrets in Colab's key panel, not in these cells. Enable access for `HF_TOKEN` and `OPENCODE_API_KEY`. The HF token must be able to read component models and write to both the private checkpoint and final-model repositories. The HF account must have access to [Stable Audio Open1.0](https://huggingface.co/stabilityai/stable-audio-open-1.0) before the CPU preflight passes. This notebook never accepts a model license on your behalf.

Fill the instructor eligibility receipt from provider/account evidence before formal training. OpenCode Go API readiness and the key's existence do not establish permission or quota for dataset review. The implementation stops at that gate instead of guessing. Keep its $50 cumulative ceiling.

For GPU work, replace the two `None` usage fields with the **actual current** Colab balance and CU/hour display. Re-run this cell and refresh the observation at least every12h. Changing architecture, splits or stage epochs creates a different checkpoint policy binding.

In [ ]:
CONFIG = json.loads((SOURCE_ROOT/'colab/final/campaign.json').read_text())
# Stable campaign name/work root across VM replacements.
CONFIG['model_repo'] = 'TassP0lar/SparkX2.6-MM'
CONFIG['model_repo_private'] = True
CONFIG['checkpoint_repo'] = 'TassP0lar/SparkX2.6-MM-checkpoints'
CONFIG['checkpoint_repo_private'] = True
CONFIG['drive_project_folder'] = '1ELmWWHxpCb3IGM2LZDQV-u7NiLxK12T_'
CONFIG['observed_compute_balance'] = None  # Enter currently displayed balance before GPU commands.
CONFIG['observed_cu_per_hour'] = None     # Enter currently displayed active rate (do not assume a quoted price).
# Example SHAPE only: replace with actual provider/account evidence. Never invent an approval.
# CONFIG['instructor_route_eligibility_receipt'] = {
#   'provider':'OpenCode Go', 'model':'mimo-v2.6-pro',
#   'endpoint':'https://opencode.ai/inference/go/openai/v1/chat/completions',
#   'permits_dataset_review':True, 'evidence_url':'PROVIDER_EVIDENCE', 'checked_at':'ACTUAL_DATE'
# }
WORK = pathlib.Path(CONFIG['work_root'])
assert WORK.resolve().is_relative_to('/content') and 'drive' not in WORK.parts
WORK.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = WORK/'operator-config.json'
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))
print({k:CONFIG[k] for k in ('campaign_id','model_repo','checkpoint_repo','max_compute_units','instructor_budget_usd')})


## Google access and in-memory Secrets

Authorize this reviewed notebook for the intended Drive project. Authentication uses Colab's normal credential handling; credentials are never copied into a checkpoint or model upload. Optional Drive mounting below is for browsing cold results only.

In [ ]:
from google.colab import auth, userdata
# The returned secrets remain in kernel memory. Do not print these variables.
auth.authenticate_user()
HF_SECRET = userdata.get('HF_TOKEN')
try:
    INSTRUCTOR_SECRET = userdata.get('OPENCODE_API_KEY')
except Exception:
    INSTRUCTOR_SECRET = None
assert HF_SECRET, 'Enable notebook access to HF_TOKEN in Colab Secrets'
print('Google authentication finished; secret presence checked without displaying values.')


In [ ]:
# Optional cold-storage browsing. No training path uses this mount.
MOUNT_DRIVE_FOR_BROWSING = False
if MOUNT_DRIVE_FOR_BROWSING:
    from google.colab import drive
    drive.mount('/content/drive')


## Install the G4 environment once

Start with G4 selected. This cell installs one isolated `env-cuda` and runs the native CUDA/Studio health check. All subsequent commands use this environment, including metadata preflight. It does not replace the notebook kernel's PyTorch. A hash-verified `uv` installer creates a pipless environment and synchronizes the complete package lock. Matching packages are reused on a retry. The source-only `sox==1.5.0` dependency is the sole source-build exception; its source archive and isolated build dependencies remain hash-checked. Other packages must have wheels. Installer stdout/stderr are captured just like campaign output.

Campaign stdout and stderr stream into this cell, with secret values and signed URL queries removed. Each invocation also saves a bounded sanitized log under `/content/spark-formal/logs`. A failure reports its exit code and actual reason. If the reason is `GatedRepo`, review the Stable Audio access form using the HF account that owns your token; once access is granted, rerun this cell without resetting the VM. A fine-grained token must also permit reading that gated model.

In [ ]:
import importlib.util, subprocess, shutil
assert sys.version_info[:2] == (3,13), 'This reviewed lock targets CPython3.13; qualify another release if Colab changes Python'
# Load only this release's standard-library helper, even after an older setup.
_spec = importlib.util.spec_from_file_location('spark_environment_bootstrap', SOURCE_ROOT/'colab/final/bootstrap.py')
_bootstrap = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_bootstrap)
assert shutil.which('nvidia-smi'), 'Select G4 in Runtime > Change runtime type before setup'
if not shutil.which('sox'):
    _bootstrap._run_checked(['apt-get','update','-qq'], WORK, _bootstrap._environment(WORK), 'apt-update')
    _bootstrap._run_checked(['apt-get','install','-y','-qq','sox','libsndfile1'], WORK, _bootstrap._environment(WORK), 'apt-audio')
CUDA_PYTHON = _bootstrap.install_environment(WORK, SOURCE_ROOT, 'cuda', base_python=sys.executable)
_spec = importlib.util.spec_from_file_location('spark_campaign_launcher', SOURCE_ROOT/'colab/final/launcher.py')
_launcher = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_launcher)

def run_campaign(mode, gpu=False, profile_receipt=None, resume_receipt=None, checkpoint_directory=None):
    python = str(WORK/'env-cuda/bin/python')
    args = [python,'-m','colab.final.run',mode,'--config',str(CONFIG_PATH),'--source-sha',SOURCE_SHA256]
    for option, value in (('--profile-receipt',profile_receipt),('--resume-receipt',resume_receipt)):
        if value is not None:
            if not isinstance(value,dict):
                raise ValueError(option+' needs the complete receipt JSON object, not a Drive folder ID')
            receipt_path = WORK/(option[2:]+'.json')
            receipt_path.write_text(json.dumps(value))
            args += [option,str(receipt_path)]
    if checkpoint_directory is not None:
        args += ['--checkpoint-directory',str(checkpoint_directory)]
    env = os.environ.copy()
    env['PYTHONPATH'] = str(SOURCE_ROOT)
    log_path = WORK/'logs'/(mode+'-'+__import__('uuid').uuid4().hex+'.log')
    return _launcher.run_process(args, cwd=SOURCE_ROOT, env=env,
        secrets={'HF_TOKEN':HF_SECRET,'OPENCODE_API_KEY':INSTRUCTOR_SECRET}, log_path=log_path)

run_campaign('doctor', gpu=True)


## Preflight and qualification on this G4

Keep the current G4 and run metadata/access preflight below. It does not allocate model weights or make instructor calls. Record the current usage balance/rate before qualification. This release targets SM120, at least90GB GPU memory and60GB RAM. It fails on a different assignment instead of silently switching to LoRA or low-bit training.

The actual Studio backend ships inside the pinned Unsloth wheel. Its full graphical UI extra conflicts with the reviewed Transformers version, so this campaign uses Studio's real headless training backend. Torch SDPA, BF16 compute, gradient checkpointing and AdamW8bit are active; custom Spark speed is measured, not assumed. This cell may take time and incurs runtime charges while a GPU is attached.

In [ ]:
# CPU work inside the same CUDA environment; no runtime or environment switch.
run_campaign('cpu')


In [ ]:
RUN_QUALIFICATION = False  # Run once on the first GPU VM; do not repeat after it passes.
if RUN_QUALIFICATION:
    assert CONFIG['observed_compute_balance'] and CONFIG['observed_cu_per_hour'], 'Refresh and rerun operator configuration'
    run_campaign('qualify', gpu=True)
    print('Qualification finished. Keep this G4; the next cell loads its profile receipt automatically.')


## Formal training on this G4, and later resume

After qualification, leave `PROFILE_RECEIPT=None` and `RESUME_RECEIPT=None`, then select `RUN_MODE='train'`. The profile receipt is read from this VM automatically. `train` launches a new subprocess, restores the complete qualification checkpoint and reproduces two native continuation updates against its saved optimizer/RNG oracle. It then restores the original initial snapshot and begins formal stages. Keep the same G4. The proof records `fresh_vm=False` when the VM did not change.

After a later VM loss, paste the complete **qualification-profile receipt JSON object**, and for `resume` also the complete **HF checkpoint receipt JSON object**. Neither field takes a Drive folder ID. A profile does not exist until qualification succeeds; a preflight receipt cannot replace it.

If a later session ends, use `resume` with the last exact **HF checkpoint receipt** (not a profile/evaluation/unfinished upload). The receipt pins a commit and complete manifest; loading a moving `main` branch is not a resume. Weights, optimizer, scheduler, RNG, stage and optimizer-committed data cursor are restored. Instructor intents are reconciled from Drive; an ambiguous POST requires operator/provider reconciliation. The $50 ceiling includes accumulated provider cash/list/allowance exposure.

The full-stage projection must fit the current authorized CU balance plus reserve. Watch `training`, `hf_checkpoint_committed`, `local_checkpoint_purged`, `qualification_complete` and `final_model_uploaded` events. A stop event is not completion. The entire six-stage curriculum and two controlled RSI rounds are configured below the launcher in the versioned runtime.

In [ ]:
PROFILE_RECEIPT = None  # Auto-loads this VM's profile; paste its complete receipt dict only after VM loss.
RESUME_RECEIPT = None   # For a later interruption, paste an exact committed checkpoint receipt dict.
RUN_MODE = 'inspect'    # 'inspect', 'train', 'resume', or 'recover' (recovery only).
if RUN_MODE == 'inspect':
    print('No GPU training launched. Select a mode explicitly when ready.')
elif RUN_MODE in ('train','resume','recover'):
    if PROFILE_RECEIPT is None and (WORK/'QUALIFICATION_RECEIPT.json').exists():
        PROFILE_RECEIPT = json.loads((WORK/'QUALIFICATION_RECEIPT.json').read_text())
    assert isinstance(PROFILE_RECEIPT,dict), 'Finish qualification first, or paste its complete receipt JSON object; a Drive folder ID is not a receipt'
    if RUN_MODE == 'resume':
        assert isinstance(RESUME_RECEIPT,dict), 'Resume needs the complete verified HF checkpoint receipt JSON object'
    assert CONFIG['observed_compute_balance'] and CONFIG['observed_cu_per_hour'], 'Refresh current usage observations'
    if RUN_MODE != 'recover':
        assert CONFIG['instructor_route_eligibility_receipt'], 'Resolve the provider/account dataset-review eligibility before formal training'
        assert INSTRUCTOR_SECRET, 'Enable OPENCODE_API_KEY'
    run_campaign(RUN_MODE,gpu=True,profile_receipt=PROFILE_RECEIPT,resume_receipt=RESUME_RECEIPT)
else:
    raise ValueError('Unknown explicit run mode')


## Checkpoint discovery and upload-only recovery

Use this cell only after the training process has exited. If an HF upload or its small Drive receipt failed, keep the VM and its local checkpoint. Enter the exact closed directory from the stopped run, then retry its upload in this same environment. This performs no optimizer steps or teacher calls. Success verifies every remote file, backs up the receipt and purges older verified local checkpoints; use the resulting receipt with `RUN_MODE='resume'`.

The HF branch updates `checkpoint/` atomically. Earlier exact commits remain recoverable for qualification and RSI; the script never squashes their history. Updating a branch does **not** reclaim HF history quota. HF storage/access/quota failures stop cleanup and preserve local files. Old local snapshots, not remote history, are purged.

In [ ]:
DISCOVER_CHECKPOINTS = False
RETRY_CHECKPOINT_DIRECTORY = None  # Example: str(WORK/'checkpoints/joint_full-00000100')
if DISCOVER_CHECKPOINTS:
    run_campaign('discover')
if RETRY_CHECKPOINT_DIRECTORY is not None:
    run_campaign('upload-checkpoint', checkpoint_directory=RETRY_CHECKPOINT_DIRECTORY)
print('Local checkpoint directories:', [p.name for p in (WORK/'checkpoints').glob('*') if p.is_dir()])


## Receipts and results

A finished campaign uploads the selected **trained private model** using `HF_TOKEN`, verifies the remote model LFS checksum and writes `MODEL_UPLOAD_RECEIPT.json` to local disk and cold Drive. The export is also reloaded from only its own saved configs/code/weights before uploading. It excludes optimizer state, credentials, raw training data and private instructor ledgers. Complete resumable checkpoints remain in their separate private HF repository; their small receipts and reports remain in Drive.

The card reports actual internal results, the selected/rejected RSI candidates and pending human audio review. A completed curriculum is not proof of broad vision/video/music capability. Listen to the final native samples and their zero-bridge controls; examine input ablations, temporal reversal, bilingual retention and independent-group counts. Stronger downstream acceptance still needs the documented domain/public benchmarks and qualified listeners.

In [ ]:
# Read only sanitized status/receipts. Never display credentials or upload-state files.
for name in ('PREFLIGHT.json','ENVIRONMENT.json','FULL_RECOVERY_PROOF.json','MODEL_UPLOAD_RECEIPT.json'):
    path=WORK/name
    if path.exists():
        value=json.loads(path.read_text())
        if name=='PREFLIGHT.json': value={k:v for k,v in value.items() if k not in ('context','drive_quota')}
        print(name, json.dumps(value,indent=2))
# To find cold commits after VM loss, with configuration/auth/setup completed:
# run_campaign('discover')
